# EDA — Cats vs Dogs

Thin driver. All plotting logic lives in `src/eda.py` so it is importable and
testable; this notebook exists because M1 asks for notebooks under version
control, and to hold the interpretation alongside the figures.

Run top-to-bottom after `bash scripts/download.sh` and `dvc repro preprocess`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src import eda
from src.data import CLASS_NAMES, IMG_SIZE, RANDOM_STATE, discover_images, load_corrupt_list
from src.preprocess import SPLITS, read_manifest

eda._setup_style()
print(f"IMG_SIZE={IMG_SIZE}  CLASS_NAMES={CLASS_NAMES}  RANDOM_STATE={RANDOM_STATE}")

## 1. What is actually in the dataset

Counts come from a filesystem walk, not from documentation — the archive has
been republished in more than one directory shape.

In [ ]:
items = discover_images()
corrupt = load_corrupt_list()
print(f"candidate image files: {len(items):,}")
print(f"corrupt / unreadable:  {len(corrupt):,}  (from data/corrupt_files.txt)")
print(f"usable:                {len(items) - len(corrupt):,}")

for split in SPLITS:
    _, labels = read_manifest(split)
    dogs = 100 * sum(labels) / len(labels)
    print(f"  {split:5s} n={len(labels):>6,}  dog={dogs:.1f}%  cat={100 - dogs:.1f}%")

### Corrupt files matter here

This dataset family ships truncated and zero-byte JPEGs. They are excluded when
the split manifests are built, *not* skipped during training — a `try/except`
inside the training loop would fail partway through an epoch and waste the run.

## 2. Class balance and stratification

Left panel: the dataset is balanced at source, so accuracy is a meaningful
headline metric and no class weighting is needed. Right panel: the stratified
80/10/10 split preserved that balance in every split — the property audit
check 46 verifies numerically.

In [ ]:
eda.plot_class_balance()

## 3. Samples as the model sees them

Shown *after* pre-processing (224x224 RGB, scaled to [0,1]) rather than as raw
files, because the resize is lossy and what matters is the tensor the network
receives. Source images vary widely in aspect ratio, so the square resize
distorts some subjects — an acceptable trade for a fixed-size CNN input, and
the reason horizontal flip is a safe augmentation while vertical is not.

In [ ]:
eda.plot_sample_grid()

## 4. Augmentation

One image through the stack several times. The printed output range is the point
of this figure, not decoration: `RandomContrast` and `RandomZoom` were pushing
values to 1.02, outside the [0,1] contract that `build_transfer_model`'s
Rescaling layer assumes when mapping to MobileNetV2's [-1,1] range. A clip layer
was added; the range should now top out at exactly 1.000.

In [ ]:
eda.plot_augmentation_grid()

## 5. What this implies for modelling

1. **Balanced classes** → accuracy is a fair headline metric; still report
   precision/recall/F1/AUC, since a pet-adoption intake filter cares about both
   error directions.
2. **Wide aspect-ratio variation** → the square resize distorts subjects, which
   argues for augmentation that mimics framing variation (zoom, small rotation)
   rather than colour tricks.
3. **Greyscale and RGBA files present** → the unconditional RGB conversion in
   `load_image` is load-bearing, not defensive. `tests/test_preprocess.py`
   asserts it against committed fixtures.
4. **Corrupt files present** → excluded at manifest time; the count is recorded
   in `data/corrupt_files.txt` so the usable-N is reproducible.

Next: `python -m src.train` (Day 3) and `python -m src.cross_validate` (Day 4).